# Tokenisation: Where African Languages Get Expensive

**Computational Text Analysis Pack · Module 25 — Transformers and Attention**

*Technique:* Subword tokenisation; fertility and cost across languages

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/12-neural-foundations/25b_tokenisation_and_african_languages.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Transformers do not read words. They read **subword tokens** produced by BPE or
SentencePiece, learned from the pretraining corpus. Text in a language that
corpus under-represents gets shattered into more pieces.

That is not a cosmetic issue. More tokens means more cost, less text inside the
context window, and — as module 23 showed on Yorùbá names — worse accuracy.
Module 3 called this the ASCII trap in the character domain; this is the same
inequity one level up.

## The tooling

**Keras** (on TensorFlow) for the models here — its `Sequential` API is the most
readable way to state an architecture, which is why teaching material reaches
for it. **PyTorch** is the research default and what almost every paper you read
will use; the concepts transfer directly, only the syntax differs.

Both are installed here. Nothing in these two modules needs a GPU: the models
are a few hundred thousand parameters and train on CPU in under a minute.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
from transformers import AutoTokenizer

# Two tokenisers: one general-purpose, one adapted to African languages.
TOKENISERS = {
    "xlm-roberta (general)": "FacebookAI/xlm-roberta-base",
    "afro-xlmr (adapted)": "Davlan/afro-xlmr-base",
}
loaded = {}
for label, repo in TOKENISERS.items():
    try:
        loaded[label] = AutoTokenizer.from_pretrained(repo)
        print(f"loaded {label}")
    except Exception as exc:
        print(f"could not load {label}: {str(exc)[:70]}")

## Fertility: tokens per word

The standard measure. A fertility of 1.0 means one token per word; 3.0 means the
tokeniser is shattering the language. English typically sits near 1.2 for these
models — the languages to watch are the ones far above it.

In [ ]:
import numpy as np

rows = []
for lang, g in df.groupby("language"):
    text = " ".join(g["text"].head(40).astype(str))[:20000]
    words = len(tokenise(text))
    if words < 500:
        continue
    row = {"language": lang, "words": words}
    for label, tok in loaded.items():
        row[label] = len(tok.tokenize(text)) / words
    rows.append(row)

fert = pd.DataFrame(rows)
if loaded:
    main = list(loaded)[0]
    fert = fert.sort_values(main, ascending=False)
    print(fert.to_string(index=False, float_format=lambda v: f"{v:.2f}"))
    print()
    worst = fert.iloc[0]; best = fert.iloc[-1]
    print(f"{worst['language']} costs {worst[main]/best[main]:.1f}x more tokens "
          f"per word than {best['language']} under {main}.")

In [ ]:
if loaded and len(fert):
    fig, ax = plt.subplots(figsize=(8.5, 5))
    y = np.arange(len(fert))
    for j, label in enumerate(loaded):
        ax.barh(y + (j - 0.5) * 0.4, fert[label], height=0.4, label=label)
    ax.set_yticks(y); ax.set_yticklabels(fert["language"], fontsize=8)
    ax.axvline(1.0, ls="--", color="grey", lw=1)
    ax.set_xlabel("tokens per word (fertility)")
    ax.set_title("Where the tokeniser shatters the language")
    ax.legend(fontsize=8)
    plt.tight_layout()

## What this costs, concretely

Fertility is a bill and a constraint, not a curiosity.

In [ ]:
if loaded and len(fert):
    main = list(loaded)[0]
    CONTEXT = 4096
    print(f"{'language':<18}{'fertility':>11}{'words in a 4k window':>24}")
    print("-" * 54)
    for _, r in fert.iterrows():
        print(f"{r['language']:<18}{r[main]:>11.2f}{CONTEXT / r[main]:>24,.0f}")
    print()
    print("Same context window, very different amounts of actual text. A")
    print("researcher working in a high-fertility language pays more per")
    print("document and fits less of it in — for the same model and the same fee.")

In [ ]:
# What shattering looks like, word by word.
PROBE = ["government", "ijọba", "gwamnati", "serikali", "Adéyẹmí", "Ọ̀yọ́"]
if loaded:
    for label, tok in loaded.items():
        print(f"\n{label}")
        for w in PROBE:
            pieces = tok.tokenize(w)
            print(f"   {w:<14}{len(pieces):>3} -> {pieces}")
    print()
    print("Words with diacritics fragment hardest. That is the mechanism behind")
    print("the Yoruba NER failure in module 23c: the model never saw the name as")
    print("a unit, so it could not label it as one.")

### 🔧 Your turn 1

Add words from a language you work in and compare fertility across both tokenisers. Does the adapted model help?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

The adapted tokeniser usually helps for languages in its training set and does
nothing — occasionally slightly worse — for those outside it, which is the whole
inequity in one measurement. It is also why "use an African model" is not by
itself a solution: adaptation covers the languages someone had data for, and the
languages with least data benefit least. Report the tokeniser and its fertility
on your corpus alongside your results; it is as load-bearing as the model choice
and almost never mentioned.

</details>